# Baseline Models — PhenoCrop v2

Trains **6 baseline models** on the same `pheno_crop_v2` dataset pipeline used in models 2, 3, and 4.

| # | Model | Type |
|---|---|---|
| B1 | Random Forest | Classical ML |
| B2 | XGBoost | Classical ML |
| B3 | LSTM / GRU / BiLSTM | Recurrent DL |
| B4 | TempCNN | Convolutional DL |
| B5 | TCN (Temporal Convolutional Network) | Convolutional DL |
| B6 | Vanilla Transformer | Attention DL |

**Identical pipeline to model_3 / model_4:**
- Features: S1 (VV, VH, VH_VV_Ratio) + S2 (NDVI, NDWI, NDRE, EVI, SAVI, MSAVI, NDMI, GNDVI, cloud_pct)
- Plot-level leakage-aware GroupShuffleSplit (70/15/15)
- SEED=42, LOOKBACK=90 days, MAX_T=30 timesteps
- Evaluation: macro-F1, accuracy, classification report


## Cell 1 — Colab Mount (optional)

In [ ]:
import os

USE_COLAB_MOUNT = True  # Set True only on Colab

if USE_COLAB_MOUNT:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Colab mount skipped: {e}")

Mounted at /content/drive


## Cell 2 — Imports & Config

In [ ]:
from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── USER CONFIG ──────────────────────────────────────────────────────────────
DATASET_ROOT   = "/content/drive/MyDrive/pheno_crop_v2"   # Colab path
LOCAL_FALLBACK = Path("./dataset/pheno_crop_v2")          # local fallback
OUTPUT_DIR     = Path("./models")
# ─────────────────────────────────────────────────────────────────────────────

root = Path(DATASET_ROOT)
if not root.exists() and LOCAL_FALLBACK.exists():
    root = LOCAL_FALLBACK.resolve()
    print(f"Using local fallback: {root}")

s1_dir  = root / "sentinel_1"
s2_dir  = root / "sentinel_2"
gt_path = root / "ground_truth.csv"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"gt exists: {gt_path.exists()} | CUDA: {torch.cuda.is_available()}")

# ── Feature columns (identical to model_3 / model_4) ─────────────────────────
S1_COLS = ["VV", "VH", "VH_VV_Ratio"]
S2_COLS = ["NDVI", "NDWI", "NDRE", "EVI", "SAVI", "MSAVI", "NDMI", "GNDVI", "cloud_pct"]
ALL_COLS = S1_COLS + S2_COLS   # 12 features total

LOOKBACK = 90    # days of lookback window
MAX_T    = 30    # max timesteps per sample

gt exists: True | CUDA: True


## Cell 3 — Labels & Plot-Level Split

**Identical split logic to model_3 / model_4** — ensures fair comparison.

In [ ]:
gt = pd.read_csv(gt_path)
gt["date"] = pd.to_datetime(gt["date"])

stage_names  = sorted(gt["stage_name"].dropna().unique())
stage_to_idx = {n: i for i, n in enumerate(stage_names)}
idx_to_stage = {i: n for n, i in stage_to_idx.items()}
gt["stage_idx"] = gt["stage_name"].map(stage_to_idx)
num_classes = len(stage_names)
print(f"Classes ({num_classes}): {stage_to_idx}")

# Plot-level leakage-aware split (same as model_3 / model_4)
plot_df = pd.DataFrame({"plot_id": sorted(gt["plot_id"].unique())})

gss = GroupShuffleSplit(1, test_size=0.15, random_state=SEED)
tv_idx, te_idx = next(gss.split(plot_df, groups=plot_df["plot_id"]))
tv_df      = plot_df.iloc[tv_idx].reset_index(drop=True)
test_plots = set(plot_df.iloc[te_idx]["plot_id"])

gss2 = GroupShuffleSplit(1, test_size=0.1765, random_state=SEED)
tr_idx, va_idx = next(gss2.split(tv_df, groups=tv_df["plot_id"]))
train_plots = set(tv_df.iloc[tr_idx]["plot_id"])
val_plots   = set(tv_df.iloc[va_idx]["plot_id"])

assert train_plots.isdisjoint(val_plots) and train_plots.isdisjoint(test_plots)

train_rows = gt[gt["plot_id"].isin(train_plots)]
val_rows   = gt[gt["plot_id"].isin(val_plots)]
test_rows  = gt[gt["plot_id"].isin(test_plots)]
print(f"Plots  train={len(train_plots)} val={len(val_plots)} test={len(test_plots)}")
print(f"Rows   train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")

Classes (5): {'Bare': 0, 'Growth': 1, 'Ripening': 2, 'Seedling': 3, 'Tillering': 4}
Plots  train=142 val=31 test=31
Rows   train=16330 val=3565 test=3565


## Cell 4 — Dataset Class (shared by all DL models)

Returns flat sequences `[MAX_T, 12]` for RNN/CNN/Transformer models.

In [ ]:
class PhenoCropDataset(Dataset):
    """Returns per-timestep concatenated S1+S2 features for baseline DL models."""

    def __init__(self, gt_df, s1_dir, s2_dir):
        self.gt = gt_df.sort_values(["plot_id", "date"]).reset_index(drop=True)
        self.s1_cache, self.s2_cache = {}, {}
        for pid in sorted(self.gt["plot_id"].unique()):
            p1 = Path(s1_dir) / f"plot_{pid}_sar.csv"
            p2 = Path(s2_dir) / f"plot_{pid}_indices.csv"
            if p1.exists():
                d = pd.read_csv(p1); d["date"] = pd.to_datetime(d["date"])
                self.s1_cache[pid] = d
            if p2.exists():
                d = pd.read_csv(p2); d["date"] = pd.to_datetime(d["date"])
                self.s2_cache[pid] = d

    def __len__(self): return len(self.gt)

    def _window(self, df, target_date, cols):
        feats = np.zeros((MAX_T, len(cols)), dtype=np.float32)
        valid = np.zeros(MAX_T, dtype=bool)
        if df is None or df.empty:
            return feats, valid
        for c in cols:
            if c not in df.columns: df[c] = 0.0
        start = target_date - pd.Timedelta(days=LOOKBACK)
        win = df[(df["date"] > start) & (df["date"] <= target_date)].sort_values("date")
        if win.empty:
            return feats, valid
        n = min(len(win), MAX_T)
        feats[-n:] = win[cols].fillna(0.0).to_numpy(dtype=np.float32)[-n:]
        valid[-n:] = True
        return feats, valid

    def __getitem__(self, idx):
        row   = self.gt.iloc[idx]
        pid   = int(row["plot_id"])
        tdate = row["date"]
        label = int(row["stage_idx"])

        s1f, s1v = self._window(self.s1_cache.get(pid), tdate, S1_COLS)
        s2f, s2v = self._window(self.s2_cache.get(pid), tdate, S2_COLS)

        # Concatenate S1+S2 → [MAX_T, 12]
        feats = np.concatenate([s1f, s2f], axis=-1)   # [T, 12]
        valid = s1v | s2v                              # [T] any valid

        return {
            "feats": torch.tensor(feats),              # [T, 12]
            "mask":  torch.tensor(~valid),             # [T] True=missing
            "label": torch.tensor(label).long(),
        }


train_ds = PhenoCropDataset(train_rows, s1_dir, s2_dir)
val_ds   = PhenoCropDataset(val_rows,   s1_dir, s2_dir)
test_ds  = PhenoCropDataset(test_rows,  s1_dir, s2_dir)
print(f"Datasets: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")

s = train_ds[0]
print(f"  feats: {tuple(s['feats'].shape)}  mask: {tuple(s['mask'].shape)}")

Datasets: 16330 / 3565 / 3565
  feats: (30, 12)  mask: (30,)


## Cell 5 — Flat Feature Extraction (for Classical ML models)

For Random Forest and XGBoost: flatten `[T, 12]` → `[T*12]` mean-pooled vector.

In [ ]:
def extract_flat_features(gt_df, s1_dir, s2_dir):
    """Extract mean-pooled flat features and labels for classical ML models."""
    s1_cache, s2_cache = {}, {}
    for pid in sorted(gt_df["plot_id"].unique()):
        p1 = Path(s1_dir) / f"plot_{pid}_sar.csv"
        p2 = Path(s2_dir) / f"plot_{pid}_indices.csv"
        if p1.exists():
            d = pd.read_csv(p1); d["date"] = pd.to_datetime(d["date"])
            s1_cache[pid] = d
        if p2.exists():
            d = pd.read_csv(p2); d["date"] = pd.to_datetime(d["date"])
            s2_cache[pid] = d

    X, y = [], []
    for _, row in gt_df.iterrows():
        pid   = int(row["plot_id"])
        tdate = row["date"]
        label = int(row["stage_idx"])

        s1_df = s1_cache.get(pid)
        s2_df = s2_cache.get(pid)

        row_feats = []
        for df, cols in [(s1_df, S1_COLS), (s2_df, S2_COLS)]:
            if df is not None and not df.empty:
                for c in cols:
                    if c not in df.columns: df[c] = 0.0
                start = tdate - pd.Timedelta(days=LOOKBACK)
                win = df[(df["date"] > start) & (df["date"] <= tdate)]
                if not win.empty:
                    vals = win[cols].fillna(0.0).to_numpy(dtype=np.float32)
                    row_feats.append(vals.mean(axis=0))  # mean-pool over time
                    row_feats.append(vals.std(axis=0))   # std-pool over time
                    row_feats.append(vals[-1])            # last observation
                else:
                    row_feats.append(np.zeros(len(cols)))
                    row_feats.append(np.zeros(len(cols)))
                    row_feats.append(np.zeros(len(cols)))
            else:
                row_feats.append(np.zeros(len(cols)))
                row_feats.append(np.zeros(len(cols)))
                row_feats.append(np.zeros(len(cols)))

        X.append(np.concatenate(row_feats))
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


print("Extracting flat features for classical ML...")
X_train, y_train = extract_flat_features(train_rows, s1_dir, s2_dir)
X_val,   y_val   = extract_flat_features(val_rows,   s1_dir, s2_dir)
X_test,  y_test  = extract_flat_features(test_rows,  s1_dir, s2_dir)
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

Extracting flat features for classical ML...
X_train: (16330, 36)  X_val: (3565, 36)  X_test: (3565, 36)


## Cell 6 — Helper: Evaluate & Report

In [ ]:
def evaluate_sklearn(name, model, X_test, y_test, idx_to_stage):
    """Evaluate a scikit-learn / XGBoost model."""
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average="macro", zero_division=0)
    print(f"\n{'='*60}")
    print(f"  {name} — Test Results")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  Macro-F1 : {f1:.4f}")
    print(f"{'='*60}")
    labels_list = [idx_to_stage[i] for i in sorted(idx_to_stage)]
    print(classification_report(y_test, preds, target_names=labels_list, zero_division=0))
    return f1


def evaluate_dl(name, model, loader, device, idx_to_stage):
    """Evaluate a PyTorch DL model."""
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            feats  = batch["feats"].to(device)
            mask   = batch["mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(feats, mask)
            preds  = logits.argmax(dim=1)
            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    f1  = f1_score(all_true, all_pred, average="macro", zero_division=0)
    print(f"\n{'='*60}")
    print(f"  {name} — Test Results")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  Macro-F1 : {f1:.4f}")
    print(f"{'='*60}")
    labels_list = [idx_to_stage[i] for i in sorted(idx_to_stage)]
    print(classification_report(all_true, all_pred, target_names=labels_list, zero_division=0))
    return f1


def class_weights_tensor(ds, nc):
    labels = [ds[i]["label"].item() for i in range(len(ds))]
    counts = torch.bincount(torch.tensor(labels), minlength=nc).clamp(min=1).float()
    return (len(labels) / (nc * counts)).clamp(max=10.0)


def train_dl(name, model, train_loader, val_loader, cw,
             epochs=80, lr=1e-3, patience=12):
    """Generic DL training loop with early stopping on val macro-F1."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[{name}] Training on {device}...")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=cw.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "max", factor=0.5, patience=5
    )

    best_f1, wait, best_state = -1.0, 0, None

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        tr_loss, tr_true, tr_pred = 0.0, [], []
        for batch in train_loader:
            feats  = batch["feats"].to(device)
            mask   = batch["mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(feats, mask)
            loss   = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item() * labels.size(0)
            tr_true.extend(labels.cpu().numpy())
            tr_pred.extend(logits.argmax(1).detach().cpu().numpy())

        tr_f1 = f1_score(tr_true, tr_pred, average="macro", zero_division=0)

        # ── Validate ──
        model.eval()
        va_true, va_pred = [], []
        with torch.no_grad():
            for batch in val_loader:
                feats  = batch["feats"].to(device)
                mask   = batch["mask"].to(device)
                labels = batch["label"].to(device)
                logits = model(feats, mask)
                va_true.extend(labels.cpu().numpy())
                va_pred.extend(logits.argmax(1).detach().cpu().numpy())

        va_f1 = f1_score(va_true, va_pred, average="macro", zero_division=0)
        scheduler.step(va_f1)

        if va_f1 > best_f1:
            best_f1, wait = va_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1

        print(
            f"Epoch {epoch:03d} | Train F1: {tr_f1:.4f} | "
            f"Val F1: {va_f1:.4f} {'*' if wait == 0 else ''}"
        )

        if wait >= patience:
            print(f"Early stopping. Best val F1: {best_f1:.4f}")
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, device


# DataLoaders (shared across DL models)
BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Batches: {len(train_loader)} / {len(val_loader)} / {len(test_loader)}")

cw = class_weights_tensor(train_ds, num_classes)
print(f"Class weights: {dict(enumerate(cw.numpy().round(3)))}")

IN_FEATURES  = len(ALL_COLS)   # 12

# Summary dict to collect all results
results = {}

Batches: 64 / 14 / 14
Class weights: {0: np.float32(1.958), 1: np.float32(0.907), 2: np.float32(1.316), 3: np.float32(0.961), 4: np.float32(0.63)}


---
## B1 — Random Forest

In [ ]:
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X_train_sc, y_train)

val_f1_rf  = f1_score(y_val,  rf.predict(X_val_sc),  average="macro", zero_division=0)
print(f"Val Macro-F1: {val_f1_rf:.4f}")

rf_f1 = evaluate_sklearn("Random Forest", rf, X_test_sc, y_test, idx_to_stage)
results["Random Forest"] = rf_f1

import joblib
joblib.dump(rf, OUTPUT_DIR / "baseline_rf.pkl")
print(f"Saved: {OUTPUT_DIR / 'baseline_rf.pkl'}")

Training Random Forest...
Val Macro-F1: 0.6918

  Random Forest — Test Results
  Accuracy : 73.80%
  Macro-F1 : 0.7205
              precision    recall  f1-score   support

        Bare       0.28      0.73      0.40       332
      Growth       0.91      0.72      0.80       786
    Ripening       0.84      0.74      0.78       427
    Seedling       0.84      0.70      0.76       887
   Tillering       0.92      0.79      0.85      1133

    accuracy                           0.74      3565
   macro avg       0.76      0.73      0.72      3565
weighted avg       0.83      0.74      0.77      3565

Saved: models/baseline_rf.pkl


---
## B2 — XGBoost

In [ ]:
print("Training XGBoost...")

# Compute sample weights for class balance
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=SEED,
    early_stopping_rounds=30,
    verbosity=0,
)

xgb_model.fit(
    X_train_sc, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val_sc, y_val)],
    verbose=False,
)

val_f1_xgb = f1_score(y_val, xgb_model.predict(X_val_sc), average="macro", zero_division=0)
print(f"Val Macro-F1: {val_f1_xgb:.4f}")

xgb_f1 = evaluate_sklearn("XGBoost", xgb_model, X_test_sc, y_test, idx_to_stage)
results["XGBoost"] = xgb_f1
xgb_model.save_model(str(OUTPUT_DIR / "baseline_xgb.json"))
print(f"Saved: {OUTPUT_DIR / 'baseline_xgb.json'}")

Training XGBoost...
Val Macro-F1: 0.6973

  XGBoost — Test Results
  Accuracy : 73.16%
  Macro-F1 : 0.7134
              precision    recall  f1-score   support

        Bare       0.27      0.73      0.40       332
      Growth       0.89      0.70      0.78       786
    Ripening       0.80      0.74      0.77       427
    Seedling       0.85      0.69      0.76       887
   Tillering       0.95      0.78      0.86      1133

    accuracy                           0.73      3565
   macro avg       0.75      0.73      0.71      3565
weighted avg       0.83      0.73      0.76      3565

Saved: models/baseline_xgb.json


---
## B3 — LSTM / GRU / BiLSTM

Set `RNN_TYPE` to one of: `'LSTM'`, `'GRU'`, `'BiLSTM'`

In [ ]:
RNN_TYPE = "BiLSTM"   # Choose: 'LSTM' | 'GRU' | 'BiLSTM'


class RNNBaseline(nn.Module):
    """
    LSTM / GRU / BiLSTM baseline.
    Input: [B, T, 12]  mask: [B, T] True=invalid
    Output: [B, num_classes]
    """

    def __init__(self, in_features, hidden_dim, num_layers,
                 num_classes, rnn_type="LSTM", dropout=0.3):
        super().__init__()
        bidirectional = rnn_type == "BiLSTM"
        cell = nn.LSTM if rnn_type in ("LSTM", "BiLSTM") else nn.GRU

        self.rnn = cell(
            input_size=in_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.norm = nn.LayerNorm(out_dim)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(out_dim, out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim // 2, num_classes),
        )

    def forward(self, x, mask):
        # x: [B, T, F]   mask: [B, T] True=invalid
        valid = (~mask).float()  # [B, T]
        out, _ = self.rnn(x)     # [B, T, H]
        out = self.norm(out)
        # Masked mean-pool over valid timesteps
        out = (out * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp(min=1)
        return self.classifier(self.drop(out))


rnn_model = RNNBaseline(
    in_features=IN_FEATURES,
    hidden_dim=128,
    num_layers=2,
    num_classes=num_classes,
    rnn_type=RNN_TYPE,
    dropout=0.3,
)
total = sum(p.numel() for p in rnn_model.parameters())
print(f"{RNN_TYPE} params: {total:,}")

BiLSTM params: 574,725


In [ ]:
rnn_model, device = train_dl(RNN_TYPE, rnn_model, train_loader, val_loader, cw)
rnn_f1 = evaluate_dl(RNN_TYPE, rnn_model, test_loader, device, idx_to_stage)
results[RNN_TYPE] = rnn_f1
torch.save(rnn_model.state_dict(), OUTPUT_DIR / f"baseline_{RNN_TYPE.lower()}.pt")
print(f"Saved: baseline_{RNN_TYPE.lower()}.pt")


[BiLSTM] Training on cuda...
Epoch 001 | Train F1: 0.5321 | Val F1: 0.6469 *
Epoch 002 | Train F1: 0.6354 | Val F1: 0.6642 *
Epoch 003 | Train F1: 0.6460 | Val F1: 0.6556 
Epoch 004 | Train F1: 0.6535 | Val F1: 0.6887 *
Epoch 005 | Train F1: 0.6572 | Val F1: 0.7100 *
Epoch 006 | Train F1: 0.6613 | Val F1: 0.6815 
Epoch 007 | Train F1: 0.6675 | Val F1: 0.6899 
Epoch 008 | Train F1: 0.6713 | Val F1: 0.6741 
Epoch 009 | Train F1: 0.6708 | Val F1: 0.6881 
Epoch 010 | Train F1: 0.6813 | Val F1: 0.6956 
Epoch 011 | Train F1: 0.6865 | Val F1: 0.6877 
Epoch 012 | Train F1: 0.6978 | Val F1: 0.7090 
Epoch 013 | Train F1: 0.7093 | Val F1: 0.7037 
Epoch 014 | Train F1: 0.7161 | Val F1: 0.6926 
Epoch 015 | Train F1: 0.7155 | Val F1: 0.7040 
Epoch 016 | Train F1: 0.7215 | Val F1: 0.7082 
Epoch 017 | Train F1: 0.7241 | Val F1: 0.6742 
Early stopping. Best val F1: 0.7100

  BiLSTM — Test Results
  Accuracy : 66.37%
  Macro-F1 : 0.6547
              precision    recall  f1-score   support

        Bar

---
## B4 — TempCNN (Temporal CNN)

Three 1-D convolution blocks with increasing receptive field, followed by global mean-pool and MLP head. Based on Pelletier et al. (2019).

In [ ]:
class TempCNN(nn.Module):
    """
    TempCNN — Pelletier et al. (2019) style.
    Three 1-D conv blocks with kernel sizes 5/5/3 + BN + dropout,
    followed by global mean-pool and MLP classification head.
    """

    def __init__(self, in_features, num_classes, channels=64, dropout=0.3):
        super().__init__()

        def conv_block(in_ch, out_ch, k):
            return nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k // 2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )

        self.encoder = nn.Sequential(
            conv_block(in_features, channels, 5),
            conv_block(channels,    channels, 5),
            conv_block(channels,    channels * 2, 3),
        )
        out_ch = channels * 2
        self.classifier = nn.Sequential(
            nn.Linear(out_ch, out_ch // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(out_ch // 2, num_classes),
        )

    def forward(self, x, mask):
        # x: [B, T, F]  → transpose to [B, F, T] for Conv1d
        x = x.transpose(1, 2)           # [B, F, T]
        x = self.encoder(x)              # [B, C, T]
        valid = (~mask).float()          # [B, T]
        # masked mean-pool
        x = (x * valid.unsqueeze(1)).sum(-1) / valid.sum(-1, keepdim=True).clamp(min=1)
        return self.classifier(x)


tempcnn = TempCNN(in_features=IN_FEATURES, num_classes=num_classes, channels=64, dropout=0.3)
total = sum(p.numel() for p in tempcnn.parameters())
print(f"TempCNN params: {total:,}")

TempCNN params: 58,245


In [13]:
tempcnn, device = train_dl("TempCNN", tempcnn, train_loader, val_loader, cw)
tcnn_f1 = evaluate_dl("TempCNN", tempcnn, test_loader, device, idx_to_stage)
results["TempCNN"] = tcnn_f1
torch.save(tempcnn.state_dict(), OUTPUT_DIR / "baseline_tempcnn.pt")
print("Saved: baseline_tempcnn.pt")


[TempCNN] Training on cuda...
Epoch 001 | Train F1: 0.5120 | Val F1: 0.6416 *
Epoch 002 | Train F1: 0.6120 | Val F1: 0.6215 
Epoch 003 | Train F1: 0.6274 | Val F1: 0.6321 
Epoch 004 | Train F1: 0.6351 | Val F1: 0.6332 
Epoch 005 | Train F1: 0.6450 | Val F1: 0.6259 
Epoch 006 | Train F1: 0.6526 | Val F1: 0.6353 
Epoch 007 | Train F1: 0.6576 | Val F1: 0.6491 *
Epoch 008 | Train F1: 0.6683 | Val F1: 0.6435 
Epoch 009 | Train F1: 0.6720 | Val F1: 0.6178 
Epoch 010 | Train F1: 0.6769 | Val F1: 0.6805 *
Epoch 011 | Train F1: 0.6834 | Val F1: 0.6333 
Epoch 012 | Train F1: 0.6861 | Val F1: 0.6669 
Epoch 013 | Train F1: 0.6835 | Val F1: 0.6412 
Epoch 014 | Train F1: 0.6923 | Val F1: 0.6259 
Epoch 015 | Train F1: 0.6977 | Val F1: 0.6925 *
Epoch 016 | Train F1: 0.6946 | Val F1: 0.6309 
Epoch 017 | Train F1: 0.6989 | Val F1: 0.6578 
Epoch 018 | Train F1: 0.7066 | Val F1: 0.6672 
Epoch 019 | Train F1: 0.7079 | Val F1: 0.6586 
Epoch 020 | Train F1: 0.7102 | Val F1: 0.6619 
Epoch 021 | Train F1: 0.7

---
## B5 — TCN (Temporal Convolutional Network)

Dilated causal convolutions with residual connections. Each block doubles the dilation to grow the receptive field exponentially.

In [14]:
class _TCNBlock(nn.Module):
    """Single dilated residual block used in TCN."""

    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv1 = nn.utils.parametrize.register_parametrization(
            nn.Conv1d(in_ch, out_ch, kernel_size,
                      padding=pad, dilation=dilation),
            "weight", nn.utils.parametrizations.weight_norm(nn.Conv1d(in_ch, out_ch, 1)).parametrizations.weight[0]
        ) if False else nn.Conv1d(in_ch, out_ch, kernel_size,
                                   padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               padding=pad, dilation=dilation)
        self.bn1  = nn.BatchNorm1d(out_ch)
        self.bn2  = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU(inplace=True)
        self.pad = pad

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)[..., :-self.pad] if self.pad else self.conv1(x)))
        out = self.drop(out)
        out = self.relu(self.bn2(self.conv2(out)[..., :-self.pad] if self.pad else self.conv2(out)))
        out = self.drop(out)
        return self.relu(out + res)


class TCN(nn.Module):
    """
    Temporal Convolutional Network with exponentially growing dilations.
    Input:  [B, T, F]  mask: [B, T]
    Output: [B, num_classes]
    """

    def __init__(self, in_features, num_classes,
                 num_channels=None, kernel_size=3, dropout=0.2):
        super().__init__()
        if num_channels is None:
            num_channels = [64, 64, 128, 128]

        layers = []
        in_ch = in_features
        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i
            layers.append(_TCNBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch

        self.network = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Linear(in_ch, in_ch // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_ch // 2, num_classes),
        )

    def forward(self, x, mask):
        x   = x.transpose(1, 2)           # [B, F, T]
        x   = self.network(x)             # [B, C, T']
        # Adjust T if convolutions changed length
        T   = mask.shape[1]
        x   = x[..., :T]                  # [B, C, T]
        valid = (~mask).float()           # [B, T]
        x = (x * valid.unsqueeze(1)).sum(-1) / valid.sum(-1, keepdim=True).clamp(min=1)
        return self.classifier(x)


tcn = TCN(
    in_features=IN_FEATURES,
    num_classes=num_classes,
    num_channels=[64, 64, 128, 128],
    kernel_size=3,
    dropout=0.2,
)
total = sum(p.numel() for p in tcn.parameters())
print(f"TCN params: {total:,}")

TCN params: 231,237


In [15]:
tcn, device = train_dl("TCN", tcn, train_loader, val_loader, cw)
tcn_f1 = evaluate_dl("TCN", tcn, test_loader, device, idx_to_stage)
results["TCN"] = tcn_f1
torch.save(tcn.state_dict(), OUTPUT_DIR / "baseline_tcn.pt")
print("Saved: baseline_tcn.pt")


[TCN] Training on cuda...
Epoch 001 | Train F1: 0.5239 | Val F1: 0.5922 *
Epoch 002 | Train F1: 0.6251 | Val F1: 0.6327 *
Epoch 003 | Train F1: 0.6332 | Val F1: 0.6478 *
Epoch 004 | Train F1: 0.6518 | Val F1: 0.6141 
Epoch 005 | Train F1: 0.6670 | Val F1: 0.6534 *
Epoch 006 | Train F1: 0.6762 | Val F1: 0.7061 *
Epoch 007 | Train F1: 0.6881 | Val F1: 0.6125 
Epoch 008 | Train F1: 0.6965 | Val F1: 0.6568 
Epoch 009 | Train F1: 0.7034 | Val F1: 0.7005 
Epoch 010 | Train F1: 0.7156 | Val F1: 0.6645 
Epoch 011 | Train F1: 0.7272 | Val F1: 0.6901 
Epoch 012 | Train F1: 0.7408 | Val F1: 0.6861 
Epoch 013 | Train F1: 0.7499 | Val F1: 0.6582 
Epoch 014 | Train F1: 0.7548 | Val F1: 0.6475 
Epoch 015 | Train F1: 0.7649 | Val F1: 0.6623 
Epoch 016 | Train F1: 0.7674 | Val F1: 0.6867 
Epoch 017 | Train F1: 0.7698 | Val F1: 0.6913 
Epoch 018 | Train F1: 0.7754 | Val F1: 0.7085 *
Epoch 019 | Train F1: 0.7772 | Val F1: 0.7030 
Epoch 020 | Train F1: 0.7810 | Val F1: 0.6609 
Epoch 021 | Train F1: 0.780

---
## B6 — Vanilla Transformer

Sinusoidal position encoding + standard Transformer encoder blocks + masked mean-pool + MLP head.

In [16]:
def sinusoid_table(n_pos, d_hid):
    pos = torch.arange(n_pos).float().unsqueeze(1)
    dim = torch.arange(d_hid).float().unsqueeze(0)
    a   = pos / torch.pow(10000, 2 * (dim // 2) / d_hid)
    a[:, 0::2] = torch.sin(a[:, 0::2])
    a[:, 1::2] = torch.cos(a[:, 1::2])
    return a


class VanillaTransformer(nn.Module):
    """
    Standard Transformer encoder baseline.
    Input projection → sinusoidal PE → L encoder layers → masked mean-pool → MLP.
    Input:  [B, T, F]  mask: [B, T] True=invalid
    Output: [B, num_classes]
    """

    def __init__(self, in_features, d_model, nhead, num_layers,
                 num_classes, dim_feedforward=256, dropout=0.1, max_len=128):
        super().__init__()
        self.input_proj = nn.Linear(in_features, d_model)
        self.register_buffer("pos_enc", sinusoid_table(max_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # Pre-LN for stability
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x, mask):
        # x: [B, T, F]   mask: [B, T] True=invalid
        B, T, _ = x.shape
        x = self.input_proj(x) + self.pos_enc[:T].unsqueeze(0)  # [B, T, d]

        # Guard: unmask last position if all masked
        kpm = mask  # True = ignore
        all_masked = kpm.all(dim=1)
        if all_masked.any():
            kpm = kpm.clone()
            kpm[all_masked, -1] = False

        x = self.encoder(x, src_key_padding_mask=kpm)           # [B, T, d]

        valid = (~kpm).float().unsqueeze(-1)                      # [B, T, 1]
        x = (x * valid).sum(1) / valid.sum(1).clamp(min=1)       # [B, d]
        return self.classifier(x)


vt = VanillaTransformer(
    in_features=IN_FEATURES,
    d_model=128,
    nhead=8,
    num_layers=3,
    num_classes=num_classes,
    dim_feedforward=256,
    dropout=0.1,
    max_len=MAX_T + 4,
)
total = sum(p.numel() for p in vt.parameters())
print(f"Vanilla Transformer params: {total:,}")

# Sanity check
dummy_x = torch.randn(4, MAX_T, IN_FEATURES)
dummy_m = torch.zeros(4, MAX_T, dtype=torch.bool)
print(f"Output shape: {vt(dummy_x, dummy_m).shape}  ✓")

Vanilla Transformer params: 407,941
Output shape: torch.Size([4, 5])  ✓


In [ ]:
vt, device = train_dl("Vanilla Transformer", vt, train_loader, val_loader, cw, lr=5e-4)
vt_f1 = evaluate_dl("Vanilla Transformer", vt, test_loader, device, idx_to_stage)
results["Vanilla Transformer"] = vt_f1
torch.save(vt.state_dict(), OUTPUT_DIR / "baseline_vanilla_transformer.pt")
print("Saved: baseline_vanilla_transformer.pt")


[Vanilla Transformer] Training on cuda...
Epoch 001 | Train F1: 0.4812 | Val F1: 0.5596 *
Epoch 002 | Train F1: 0.5840 | Val F1: 0.6205 *
Epoch 003 | Train F1: 0.6049 | Val F1: 0.6374 *
Epoch 004 | Train F1: 0.6159 | Val F1: 0.6183 
Epoch 005 | Train F1: 0.6264 | Val F1: 0.6433 *
Epoch 006 | Train F1: 0.6302 | Val F1: 0.6404 
Epoch 007 | Train F1: 0.6411 | Val F1: 0.6889 *
Epoch 008 | Train F1: 0.6517 | Val F1: 0.6578 
Epoch 009 | Train F1: 0.6503 | Val F1: 0.6850 
Epoch 010 | Train F1: 0.6647 | Val F1: 0.6449 
Epoch 011 | Train F1: 0.6662 | Val F1: 0.6590 
Epoch 012 | Train F1: 0.6722 | Val F1: 0.7031 *
Epoch 013 | Train F1: 0.6800 | Val F1: 0.6545 


---
## Summary — All Baseline Results

In [ ]:
print("\n" + "="*55)
print("  BASELINE COMPARISON — Test Macro-F1")
print("="*55)
print(f"  {'Model':<25} {'Macro-F1':>10}")
print("-"*55)
for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<25} {f1:>10.4f}")
print("="*55)

# Reference scores from deep models
refs = {
    "Model 3 (Presto-style Transformer)": 0.7534,
    "Model 4 (BiMamba + Transformer)":    0.7600,  # approx
}
print("\n  Reference Deep Models")
print("-"*55)
for name, f1 in refs.items():
    print(f"  {name:<37} {f1:>10.4f}")
print("="*55)